In [1]:
import subprocess
import os
import json
from difflib import unified_diff

In [2]:
env_packages = {}

envs = [
    r"\\org-app2\c$\Program Files\ArcGIS\Server\framework\runtime\ArcGIS\bin\Python\envs\arcgispro-py3",
    r"\\org-app2\c$\Program Files\ArcGIS\Server\framework\runtime\ArcGIS\bin\Python\envs\clone20240610",
    r"\\org-app2\c$\Program Files\ArcGIS\Server\framework\runtime\ArcGIS\bin\Python\envs\clone20240827"
]

In [3]:
def get_environment_packages(env_path):
    print(f"Getting packages in environment: {env_path}")
    try:
        # Activate the environment and list installed packages in JSON format
        cmd = f'"{os.path.join(env_path, "python.exe")}" -m pip list --format=json'
        result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, shell=True, text=True)
        
        if result.returncode != 0:
            raise RuntimeError(f"Error running pip list: {result.stderr}")
        
        # Load the package list as a dictionary
        packages = json.loads(result.stdout)
        return {pkg['name'].lower(): pkg['version'] for pkg in packages}
    except Exception as e:
        print(f"Error getting packages in environment: {env_path} - {e}")
        return {}

In [4]:
# def compare_environments(env1, env2, env1_packages, env2_packages):
def compare_environments(env1, env2):
    env1_packages = env_packages[env1]
    env2_packages = env_packages[env2]
    
    diff = None
    try:
        print(f"Comparing environments:\n1. {env1}\n2. {env2}\n")
        # Generate a unified diff
        diff = list(unified_diff(
            sorted([f"{name}=={version}" for name, version in env1_packages.items()]),
            sorted([f"{name}=={version}" for name, version in env2_packages.items()]),
            fromfile=f'env1 ({env1})',
            tofile=f'env2 ({env2})',
            lineterm=''
        ))
    except RuntimeError as e:
        print(f"An error occurred: {e}")
    if diff:
        added_or_missing = []
        version_changes = []
        
        print("\nDifferences between the two environments:\n")

        for line in diff:
            if line.startswith('-') or line.startswith('+'):
                package_info = line[1:].strip()
                if '==' in package_info:
                    name, version = package_info.split('==')
                    if name in env1_packages and name in env2_packages:
                        version_changes.append(line)
                    else:
                        added_or_missing.append(line)
                else:
                    added_or_missing.append(line)

        if added_or_missing:
            print("\nAdded or Missing Packages:\n")
            for line in added_or_missing:
                print(line)

        if version_changes:
            print("\nVersion Changes:\n")
            for line in version_changes:
                print(line)
    else:
        print("The two environments have identical installed packages.")

    return diff

In [5]:
env_packages = {env: get_environment_packages(env) for env in envs}

Getting packages in environment: \\org-app2\c$\Program Files\ArcGIS\Server\framework\runtime\ArcGIS\bin\Python\envs\arcgispro-py3
Getting packages in environment: \\org-app2\c$\Program Files\ArcGIS\Server\framework\runtime\ArcGIS\bin\Python\envs\clone20240610
Getting packages in environment: \\org-app2\c$\Program Files\ArcGIS\Server\framework\runtime\ArcGIS\bin\Python\envs\clone20240827


In [6]:
print("Comparing all environments...\n")
for i in range(len(envs)):
    for j in range(i+1, len(envs)):
        env1 = envs[i]
        env2 = envs[j]
        differences = compare_environments(env1, env2)

Comparing all environments...

Comparing environments:
1. \\org-app2\c$\Program Files\ArcGIS\Server\framework\runtime\ArcGIS\bin\Python\envs\arcgispro-py3
2. \\org-app2\c$\Program Files\ArcGIS\Server\framework\runtime\ArcGIS\bin\Python\envs\clone20240610


Differences between the two environments:


Added or Missing Packages:

--- env1 (\\org-app2\c$\Program Files\ArcGIS\Server\framework\runtime\ArcGIS\bin\Python\envs\arcgispro-py3)
+++ env2 (\\org-app2\c$\Program Files\ArcGIS\Server\framework\runtime\ArcGIS\bin\Python\envs\clone20240610)
+affine==2.4.0
+aiobotocore==2.13.0
+aiohttp==3.9.5
+aioitertools==0.11.0
+aiosignal==1.3.1
+alabaster==0.7.16
+arpeggio==2.0.2
+asciitree==0.3.3
+astroid==2.14.2
+async-timeout==4.0.3
+atomicwrites==1.4.0
+autopep8==2.0.4
+bcrypt==3.2.0
+binaryornot==0.4.4
+botocore==1.34.106
+brotli==1.0.9
+cached-property==1.5.2
+cerberus==1.3.5
+click-plugins==1.1.1
+cligj==0.7.2
+cookiecutter==2.6.0
+cramjam==2.8.3
+dephell-archive==0.1.7
+dephell-argparse==0.1.3

In [7]:
# differences = compare_environments(env1, env2, env1_packages, env2_packages)

In [8]:
# differences = compare_environments(env1, env3, env1_packages, env3_packages)

In [9]:
# differences = compare_environments(env2, env3, env2_packages, env3_packages)

In [10]:
print("\nComparison complete.")


Comparison complete.
